# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/karthikmannam/flyrank-internship-ml/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

**Binary classification with a scoring output.**

The underlying decision is a yes/no question: *"Is this page declining?"* At the row level, the model predicts `is_declining_label ∈ {0,1}` from observed content and engagement signals — that is classification.

But the output is used as a **priority score** (the predicted probability of decline), which ranks the full inventory for an editor. So the task type is classification internally, ranking by score externally. This matches the skill's "Which ones first? → Ranking / scoring" row while still using a supervised label for training.

Why not regression? The label is inherently categorical — we care about *direction* (declining or not), not the magnitude of the trend. A classifier's probability gives us a clean priority score.

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv("../../data/raw/content_refresh_anonymized.csv")
df["is_declining"] = (df["trend_direction"] == "down").astype(int)

print("Task type: Binary classification (output ranked by predicted probability)")
print()
print("Label distribution:")
print(df["is_declining"].value_counts().to_frame().rename(
    columns={"is_declining": "count"}).assign(
    pct=lambda x: (x["count"] / len(df) * 100).round(1)))
print()
n_declining = df["is_declining"].sum()
print(f"{n_declining:,} of {len(df):,} rows are declining ({n_declining/len(df)*100:.1f}%)")
print("Two classes → binary classification. Probability → ranking score.")

## 2. Target or proxy

**Target: `is_declining_label` — whether the page's recent trend direction is "down".**

This is an **observed outcome**, not a defined rule. The label comes from comparing actual impression counts in two consecutive 30-day windows (`impressions_last_30d` vs `impressions_prev_30d`): if the later window's impressions dropped by more than 20%, the label is 1 (declining). The data measures what *actually happened* to the page's search visibility.

The proxy concern: the 20% threshold is a definitional choice, but the underlying comparison of measured impressions is observed. The skill warns "The target must be observed, not defined" — here the *comparison* is observed (measured impressions from Google Search Console), and the threshold codifies what "declining" means in the client's workflow. This is acceptable as long as we never use `trend_direction` or `trend_pct` as a feature (leakage rule in the data dictionary).

In [ ]:
# Show how the label is derived from observed impression counts
df["label"] = (df["trend_direction"] == "down").astype(int)

# Show examples of the underlying measurement
provenance = df[["impressions_prev_30d", "impressions_last_30d", "trend_direction", "trend_pct", "label"]].copy()
provenance["pct_change"] = ((provenance["impressions_last_30d"] - provenance["impressions_prev_30d"]) 
                            / provenance["impressions_prev_30d"] * 100).round(1)

print("Label provenance: comparing two observed 30-day impression windows")
print(f"  Label=1 (declining) when impressions_last_30d < impressions_prev_30d by >20%")
print(f"  Label=0 (not declining) otherwise (stable, up, new, flat)")
print()
print("Example rows — label comes from observed counts, not a made-up rule:")
provenance.head(6)

## 3. Success metric

**Precision@K — specifically Precision@50.**

Why this metric: the editor has limited weekly capacity (e.g. 50 pages). Precision@50 answers the practical question: *"Of the 50 pages I recommend most urgently, how many are actually declining?"* A random baseline would get ~54% (the base rate). The starter pipeline's baseline rule got 24% (worse than random — the simple heuristic actually hurts). The starter random forest got 74%.

'Good' means statistically and operationally better than the base rate (54.2%). A useful target: Precision@50 ≥ 0.70, meaning at least 35 of the top 50 picks are truly declining — so the editor's finite time goes to pages that need it.

Recall and average precision are secondary checks, but Precision@K is the metric that maps directly to the decision.

In [ ]:
# Compute base rate and show Precision@K concept
base_rate = df["label"].mean()
print(f"Base rate (declining): {base_rate:.1%}")
print(f"Random Precision@50:   {base_rate:.1%}  (what random picks would get)")
print()
print("Starter pipeline benchmarks (from committed outputs):")
print("  Baseline rule Precision@50:  0.240 (hand-written heuristic)")
print("  Random forest Precision@50:  0.740 (learned model)")
print()
print(f"Target: Precision@50 >= 0.70 — at least 35 of the top 50 picks are truly declining.")
print(f"This means the editor's finite time goes to pages that need it.")

## 4. The unit of analysis, as a real dataframe

**One row = one content item (one page).**

The row grain is defined by `content_id` — each row is a unique page with its aggregated 90-day performance metrics, keyword context, and content properties. This is the natural unit for the editorial decision: which *page* needs attention first.

Below: a slice from the raw dataset showing the key fields for the refresh-scoring task: identifiers (grouping only), engagement signals, trend inputs, and content meta.

In [ ]:
# Show the unit of analysis: one row = one content page
unit_cols = [
    "content_id", "client_id", "content_type", "content_age_days",
    "impressions_90d", "ctr", "avg_position", "engagement_rate",
    "days_since_last_update", "impression_tier", "trend_direction",
]

unit_df = df[unit_cols].copy()
unit_df["is_declining"] = df["label"]

print(f"One row = one page (content item). Shape: {unit_df.shape}")
print(f"Units of analysis: {len(unit_df):,} pages across {unit_df['client_id'].nunique()} clients")
print()
unit_df.head(10)

## 5. Why ML beats a fixed rule here

**Decline emerges from many weak signals interacting — a hand-written rule can't capture the combinations.**

A fixed rule like *"Score = -trend_pct + (position > 10) × weight"* misses the reality that decline depends on context:
- A page with high impressions and slipping position is different from a low-impression page with the same position change.
- Content type, keyword competition, and engagement rate interact: a thin `feedly article` with dropping CTR signals something different than a well-trafficked `keyword article` with the same CTR drop.
- Some signals are only meaningful in combination: `avg_position = 0` means "no data" (1,205 rows), not rank zero — a rule that treats 0 as a rank value is wrong by construction.
- Missingness is systematic by content_type — a rule that blindly fillna(0) encodes false signals.

A learned model can weight these interactions from data instead of guessing the right thresholds. The starter pipeline proves this empirically: the rule baseline gets Precision@50 = 0.240 (worse than the 54% base rate — the simple rule actively misranks), while a random forest gets 0.740. That 50-percentage-point lift comes from finding patterns no single if-statement expresses.

In [ ]:
# Show why a fixed rule fails: same label-input value, different outcomes
import numpy as np

# Pages with identical trend_pct but opposite labels - context matters
contradict = df.groupby("trend_pct").filter(lambda g: g["label"].nunique() > 1)
contradict = contradict.groupby("trend_pct").filter(lambda g: len(g) >= 5)
print(f"Pages with identical trend_pct but both labels: {len(contradict):,} rows")
print(f"  -> A rule based on trend_pct alone will always be wrong for these")
print()

bucket = contradict[contradict["trend_pct"] == -50.0]
if len(bucket) > 0:
    print(f"At trend_pct = -50.0 ({len(bucket)} pages):")
    show = bucket[["impressions_90d", "avg_position", "ctr", "content_type", "label"]].copy()
    print(show.to_string())
    print()
    print("Same trend strength - but some declining and some aren't.")
    print("Volume, position, content type tip the balance. ML sees all of them.")
print()

# Quick ML demo: no label-derived signals, only safe features
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import precision_score

safe_features = ["ctr", "avg_position", "engagement_rate", "content_age_days",
                "impressions_90d", "days_since_last_update", "competition"]
demo_X = df[safe_features].copy()
demo_X["avg_position"] = demo_X["avg_position"].replace(0, np.nan).fillna(demo_X["avg_position"].median())
demo_X = demo_X.fillna(0)
y = df["label"]

simple = DecisionTreeClassifier(max_depth=1, random_state=42)
simple.fit(demo_X, y)
y_simple = simple.predict(demo_X)
print(f"Single-split rule (max_depth=1) Precision: {precision_score(y, y_simple):.3f}")
print(f"  Split on: {safe_features[simple.tree_.feature[0]]}")

interact = DecisionTreeClassifier(max_depth=5, random_state=42)
interact.fit(demo_X, y)
y_interact = interact.predict(demo_X)
print(f"Interactive tree (max_depth=5)    Precision: {precision_score(y, y_interact):.3f}")
print()
print("Without using the label source (trend_pct), a single split is weak.")
print("Adding depth (interactions) lifts precision - ML finds combinations.")

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime -> Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.